# 02_feature_engineering.ipynb

Construye el dataset de entrenamiento (~1.15M muestras) emitiendo `CanonicalRequest`
en formato JSONL por cada fuente y delegando el calculo de las 72 features al
extractor canonico en TypeScript (`packages/extractor`). Ver `.claude/CLAUDE.md`,
seccion "Arquitectura tecnica: extractor de features canonico (R1)".

Flujo por fuente:
1. Leer datos crudos y mapear cada fila a `CanonicalRequest` + `{sample_id, label, timestamp}`.
2. Escribir JSONL en `data/processed/canonical/<nombre>.jsonl`.
3. Invocar `node packages/extractor/dist/cli.js <jsonl> <csv>` -> CSV con 72 features.
4. Cargar el CSV, (solo OWASP/RussellMitchell) calcular Grupo 9 (temporal) por separado, y guardar parquet.

Python **no calcula features**: solo arma el JSON de entrada y consume el CSV de salida (R1).
El schema de salida es uniforme: `sample_id`, 72 features (`FEATURE_NAMES`), `label` (int), `timestamp`.

Labels:
- 0: legitimate
- 1: sqli
- 2: xss
- 3: path_traversal
- 4: command_injection

Fuentes procesadas en este notebook:
1. Payloads.csv
2. payload_full.csv
3. command_injection.csv
4. XSS_dataset.csv
5. data_capec_multilabel.csv
6. modsec-learn JSON
7. PT wordlists (Dp.txt)
8. OWASP logs
9. RussellMitchell (legitimate only)


In [1]:
import pandas as pd
import numpy as np
import re, json, html as html_lib, subprocess
from pathlib import Path
from urllib.parse import unquote, urlparse

BASE_DIR      = Path("../data")
PROCESSED_DIR = Path("../data/processed")
CANONICAL_DIR = PROCESSED_DIR / "canonical"
PROCESSED_DIR.mkdir(exist_ok=True)
CANONICAL_DIR.mkdir(exist_ok=True)

EXTRACTOR_CLI = Path("../packages/extractor/dist/cli.js")
assert EXTRACTOR_CLI.exists(), (
    f"Extractor TS no compilado: {EXTRACTOR_CLI}. "
    f"Correr `pnpm --filter extractor run build` desde la raiz del repo."
)

TODAY = pd.Timestamp("2026-06-08")

LABEL = {
    "legitimate": 0, "sqli": 1, "xss": 2,
    "path_traversal": 3, "command_injection": 4,
}

# Columnas canonicas en orden (72 features). Calculadas por packages/extractor (TS);
# se mantienen aqui solo para validacion/resumen (Seccion 12).
FEATURE_COLS = [
    # Grupo 1: longitudes
    "payload_length", "payload_entropy", "uri_length", "path_length",
    "query_string_length", "body_length", "body_entropy",
    "path_depth", "query_param_count", "fragment_present",
    # Grupo 2: composicion de caracteres
    "special_char_ratio", "numeric_char_ratio", "uppercase_ratio",
    "whitespace_count", "newline_char_count", "null_byte_count",
    "extended_ascii_ratio", "payload_token_count",
    # Grupo 3: encoding
    "url_encoded_ratio", "encoded_char_freq", "double_encoded_count",
    "hex_escape_count", "unicode_escape_count", "html_entity_count", "base64_like_count",
    # Grupo 4: SQLi
    "sqli_keyword_count", "sqli_keyword_density", "sqli_comment_count",
    "sqli_operator_count", "quote_count", "semicolon_count", "parenthesis_count",
    "union_present", "select_present",
    # Grupo 5: XSS
    "xss_marker_count", "xss_marker_density", "html_tag_count",
    "script_tag_present", "js_event_handler_count", "javascript_url_count",
    "html_entity_density", "alert_function_present", "inline_style_present",
    # Grupo 6: Path Traversal
    "traversal_sequence_count", "path_separator_count", "absolute_path_indicator",
    "sensitive_file_target", "sensitive_extension_count", "file_extension_suspicious",
    "dotdot_encoded_count",
    # Grupo 7: Command Injection
    "pipe_count", "backtick_count", "shell_command_count",
    "command_separator_count", "redirect_operator_count",
    "dollar_sign_count", "subshell_count", "os_path_indicator",
    # Grupo 8: HTTP request
    "method_is_get", "method_is_post", "ua_present", "ua_length",
    "ua_suspicious", "content_type_encoded", "authorization_length",
    "unusual_headers_count", "status_code",
    # Grupo 9: temporal (0 para fuentes sin timestamp de sesion)
    "req_count_1s", "req_count_5s", "req_count_60s",
    "error_rate_4xx_60s", "endpoint_diversity_60s",
]

assert len(FEATURE_COLS) == 72, f"Esperado 72, hay {len(FEATURE_COLS)}"
print(f"Setup OK | PROCESSED_DIR={PROCESSED_DIR} | CANONICAL_DIR={CANONICAL_DIR} | features={len(FEATURE_COLS)}")

Setup OK | PROCESSED_DIR=..\data\processed | CANONICAL_DIR=..\data\processed\canonical | features=72


# Seccion 2: Emision de CanonicalRequest e invocacion del extractor TS

`make_canonical()` construye un registro `CanonicalRequest` (mas `sample_id`/`label`/`timestamp`)
con los defaults de `.claude/CANONICAL_REQUEST_NOTES.md` seccion 6. `write_jsonl()` lo serializa,
`run_extractor()` invoca el CLI de `packages/extractor` y `load_features()` lee el CSV resultante
(`sample_id, label, timestamp` + 72 features en el orden de `FEATURE_NAMES`).

`add_temporal_features()` y `save_parquet()` se mantienen en Python: el Grupo 9 (temporal) requiere
ventanas deslizantes por IP sobre todo el dataset (estado entre requests), algo que el extractor
TS deliberadamente no calcula (es una funcion pura por-request, ver `index.ts`). Solo OWASP y
RussellMitchell tienen IP+timestamp reales; para las demas fuentes el Grupo 9 queda en 0.


In [2]:
def make_canonical(sample_id, label, timestamp=None, *, method="", path="", query="",
                    body="", user_agent="", content_type="", referer="", cookie="",
                    extra_headers=None, status_code=0):
    """Construye un registro CanonicalRequest + metadatos para el CLI TS."""
    if status_code is None or (isinstance(status_code, float) and np.isnan(status_code)):
        status_code = 0
    return {
        "sample_id": sample_id,
        "label": int(label),
        "timestamp": (timestamp if timestamp is not None else TODAY).isoformat(),
        "method": str(method or ""),
        "path": str(path or ""),
        "query": str(query or ""),
        "body": str(body or ""),
        "userAgent": str(user_agent or ""),
        "contentType": str(content_type or ""),
        "referer": str(referer or ""),
        "cookie": str(cookie or ""),
        "extraHeaders": extra_headers or {},
        "statusCode": int(status_code),
    }


def write_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  JSONL escrito: {path.name} ({len(records):,} registros)")


def run_extractor(jsonl_path, csv_path):
    """Invoca el CLI TS de packages/extractor: JSONL CanonicalRequest -> CSV de 72 features."""
    result = subprocess.run(
        ["node", str(EXTRACTOR_CLI), str(jsonl_path), str(csv_path)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"extractor CLI fallo ({jsonl_path.name}): {result.stderr}")
    print(f"  {result.stderr.strip()}")


def load_features(csv_path):
    df = pd.read_csv(csv_path)
    df["label"] = df["label"].astype("int8")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df


print("Helpers de emision canonica definidos")

Helpers de emision canonica definidos


In [3]:
def add_temporal_features(df_feat: pd.DataFrame, df_src: pd.DataFrame,
                            ts_col: str, ip_col: str,
                            endpoint_col: str, status_col: str) -> pd.DataFrame:
    """
    Calcula features temporales por IP usando ventanas deslizantes.
    Solo para fuentes con timestamps reales (OWASP, RussellMitchell).
    df_src debe tener ts_col (datetime), ip_col, endpoint_col, status_col.
    Los indices de df_feat y df_src deben estar alineados.
    """
    df_src = df_src.copy()
    df_src[ts_col] = pd.to_datetime(df_src[ts_col])
    df_src = df_src.sort_values(ts_col)

    req_1s  = np.zeros(len(df_src), dtype=np.float32)
    req_5s  = np.zeros(len(df_src), dtype=np.float32)
    req_60s = np.zeros(len(df_src), dtype=np.float32)
    err_60s = np.zeros(len(df_src), dtype=np.float32)
    div_60s = np.zeros(len(df_src), dtype=np.float32)

    ts_arr  = df_src[ts_col].values.astype("int64")  # nanoseconds
    ip_arr  = df_src[ip_col].values
    ep_arr  = df_src[endpoint_col].values
    st_arr  = df_src[status_col].fillna(0).astype(int).values

    NS = 1_000_000_000  # nanoseconds per second

    for i in range(len(df_src)):
        t_i  = ts_arr[i]
        ip_i = ip_arr[i]
        # Window: same IP, within [t_i - W, t_i)
        def _count_window(w_ns):
            lo = t_i - w_ns * NS
            mask = (ts_arr >= lo) & (ts_arr < t_i) & (ip_arr == ip_i)
            return int(mask.sum())

        req_1s[i]  = _count_window(1)
        req_5s[i]  = _count_window(5)
        req_60s[i] = _count_window(60)

        lo60 = t_i - 60 * NS
        m60  = (ts_arr >= lo60) & (ts_arr < t_i) & (ip_arr == ip_i)
        if m60.sum() > 0:
            err_60s[i] = float((st_arr[m60] // 100 == 4).sum()) / m60.sum()
            div_60s[i] = float(len(np.unique(ep_arr[m60])))

    # Reasignar al indice original de df_feat
    orig_idx = df_src.index
    for col in ("req_count_1s", "req_count_5s", "req_count_60s",
                "error_rate_4xx_60s", "endpoint_diversity_60s"):
        df_feat[col] = df_feat[col].astype("float64")
    df_feat.loc[orig_idx, "req_count_1s"]           = req_1s
    df_feat.loc[orig_idx, "req_count_5s"]           = req_5s
    df_feat.loc[orig_idx, "req_count_60s"]          = req_60s
    df_feat.loc[orig_idx, "error_rate_4xx_60s"]     = err_60s
    df_feat.loc[orig_idx, "endpoint_diversity_60s"] = div_60s
    return df_feat


def save_parquet(df: pd.DataFrame, name: str) -> Path:
    out = PROCESSED_DIR / f"{name}.parquet"
    df.to_parquet(out, index=False, engine="pyarrow")
    size_kb = out.stat().st_size / 1024
    print(f"  Guardado: {out.name}  ({len(df):,} filas, {size_kb:.0f} KB)")
    label_dist = df['label'].value_counts().sort_index().to_dict()
    inv = {v:k for k,v in LABEL.items()}
    for lbl_int, cnt in sorted(label_dist.items()):
        print(f"    label={int(lbl_int)} ({inv.get(int(lbl_int),'?'):<20}) : {cnt:>8,}")
    return out

print("Funciones auxiliares definidas")

Funciones auxiliares definidas


# Seccion 3: Payloads.csv

URLs completas etiquetadas Benign/Malicious. Malicious = XSS exclusivamente.
Encoding: latin-1. Se eliminan 546 duplicados antes de procesar.
CanonicalRequest: `path`/`query` se obtienen con `urlparse()` sobre la URL completa
(decodificados). `rawPayload` (= `query` si existe, si no `path`) preserva el
fragmento donde va el XSS. Timestamp: no disponible, se usa TODAY (2026-06-08).


In [4]:
df_pc = pd.read_csv(BASE_DIR / "Payloads.csv", encoding="latin-1")
df_pc = df_pc.drop_duplicates(subset="Payloads").reset_index(drop=True)


def _split_url(url):
    try:
        parsed = urlparse(str(url))
        return unquote(parsed.path), unquote(parsed.query)
    except Exception:
        return unquote(str(url)), ""


_split = df_pc["Payloads"].apply(_split_url)
df_pc["path"]  = _split.apply(lambda x: x[0])
df_pc["query"] = _split.apply(lambda x: x[1])

LABEL_MAP_PC = {"Benign": LABEL["legitimate"], "Malicious": LABEL["xss"]}

records = [
    make_canonical(sample_id=f"payloads_csv_{i}", label=LABEL_MAP_PC[cls], path=p, query=q)
    for i, (cls, p, q) in enumerate(zip(df_pc["Class"], df_pc["path"], df_pc["query"]))
]

jsonl_path = CANONICAL_DIR / "payloads_csv.jsonl"
csv_path   = PROCESSED_DIR / "payloads_csv_features.csv"
write_jsonl(records, jsonl_path)
run_extractor(jsonl_path, csv_path)
df_out = load_features(csv_path)

print(f"Payloads.csv | filas={len(df_out):,}")
print(df_out[["sample_id","payload_length","encoded_char_freq","xss_marker_count","label"]].head(3).to_string())
save_parquet(df_out, "payloads_csv")

  JSONL escrito: payloads_csv.jsonl (42,671 registros)


  OK: 42671 filas escritas en ..\data\processed\payloads_csv_features.csv (0 omitidas)
Payloads.csv | filas=42,671
        sample_id  payload_length  encoded_char_freq  xss_marker_count  label
0  payloads_csv_0              54                  0                 3      2
1  payloads_csv_1             142                  2                 1      2
2  payloads_csv_2              71                  0                 0      2


  Guardado: payloads_csv.parquet  (42,671 filas, 1528 KB)
    label=0 (legitimate          ) :   28,068
    label=2 (xss                 ) :   14,603


WindowsPath('../data/processed/payloads_csv.parquet')

# Seccion 4: payload_full.csv

Unico CSV con los cuatro vectores. CanonicalRequest: el payload (URL-decodificado)
se asigna a `body` (`rawPayload = body`, prioridad mas alta en `deriveRawPayload`).
Timestamp: no disponible, se usa TODAY.


In [5]:
df_pf = pd.read_csv(BASE_DIR / "payload_full.csv", encoding="utf-8")

ATTACK_MAP = {
    "norm"          : LABEL["legitimate"],
    "sqli"          : LABEL["sqli"],
    "xss"           : LABEL["xss"],
    "path-traversal": LABEL["path_traversal"],
    "cmdi"          : LABEL["command_injection"],
}

df_pf["payload_dec"] = df_pf["payload"].apply(lambda x: unquote(str(x)))

records = [
    make_canonical(sample_id=f"payload_full_{i}", label=ATTACK_MAP[atk], body=pl)
    for i, (atk, pl) in enumerate(zip(df_pf["attack_type"], df_pf["payload_dec"]))
]

jsonl_path = CANONICAL_DIR / "payload_full.jsonl"
csv_path   = PROCESSED_DIR / "payload_full_features.csv"
write_jsonl(records, jsonl_path)
run_extractor(jsonl_path, csv_path)
df_out = load_features(csv_path)

print(f"payload_full.csv | filas={len(df_out):,}")
print(df_out[["sample_id","payload_length","sqli_keyword_count","xss_marker_count","label"]].head(3).to_string())
save_parquet(df_out, "payload_full")

  JSONL escrito: payload_full.jsonl (31,067 registros)


  OK: 31067 filas escritas en ..\data\processed\payload_full_features.csv (0 omitidas)
payload_full.csv | filas=31,067
        sample_id  payload_length  sqli_keyword_count  xss_marker_count  label
0  payload_full_0              14                   0                 0      0
1  payload_full_1              12                   0                 0      0
2  payload_full_2               5                   0                 0      0
  Guardado: payload_full.parquet  (31,067 filas, 794 KB)
    label=0 (legitimate          ) :   19,304
    label=1 (sqli                ) :   10,852
    label=2 (xss                 ) :      532
    label=3 (path_traversal      ) :      290
    label=4 (command_injection   ) :       89


WindowsPath('../data/processed/payload_full.parquet')

# Seccion 5: command_injection.csv

Dataset binario CMDI. Los payloads estan HTML-encoded en varios casos:
primero se decodifica HTML (`html.unescape`), luego URL-decode. CanonicalRequest:
el resultado se asigna a `body` (`rawPayload = body`). Timestamp: no disponible, se usa TODAY.


In [6]:
df_ci = pd.read_csv(BASE_DIR / "command injection.csv", encoding="latin-1")
df_ci = df_ci.dropna(subset=["sentence"]).drop_duplicates(subset="sentence").reset_index(drop=True)

# Doble decodificacion: HTML-unescape -> URL-decode
df_ci["payload_dec"] = df_ci["sentence"].apply(
    lambda x: unquote(html_lib.unescape(str(x)))
)

LABEL_MAP_CI = {0: LABEL["legitimate"], 1: LABEL["command_injection"]}

records = [
    make_canonical(sample_id=f"cmd_injection_{i}", label=LABEL_MAP_CI[lbl], body=pl)
    for i, (lbl, pl) in enumerate(zip(df_ci["Label"], df_ci["payload_dec"]))
]

jsonl_path = CANONICAL_DIR / "command_injection.jsonl"
csv_path   = PROCESSED_DIR / "command_injection_features.csv"
write_jsonl(records, jsonl_path)
run_extractor(jsonl_path, csv_path)
df_out = load_features(csv_path)

print(f"command_injection.csv | filas={len(df_out):,}")
print(df_out[["sample_id","payload_length","shell_command_count","pipe_count","label"]].head(4).to_string())
save_parquet(df_out, "command_injection")

  JSONL escrito: command_injection.jsonl (2,059 registros)
  OK: 2059 filas escritas en ..\data\processed\command_injection_features.csv (0 omitidas)
command_injection.csv | filas=2,059
         sample_id  payload_length  shell_command_count  pipe_count  label
0  cmd_injection_0              39                    1           0      4
1  cmd_injection_1              39                    1           0      4
2  cmd_injection_2              30                    2           0      4
3  cmd_injection_3              15                    1           2      4
  Guardado: command_injection.parquet  (2,059 filas, 88 KB)
    label=0 (legitimate          ) :    1,581
    label=4 (command_injection   ) :      478


WindowsPath('../data/processed/command_injection.parquet')

# Seccion 6: XSS_dataset.csv

Dataset binario XSS. ADVERTENCIA: Label=0 son textos de Wikipedia, no trafico HTTP.
Esto se preserva en el parquet pero se documenta en la columna sample_id con prefijo.
CanonicalRequest: `Sentence` (URL-decodificado) se asigna a `body` (`rawPayload = body`).
Timestamp: no disponible, se usa TODAY.


In [7]:
df_xss = pd.read_csv(BASE_DIR / "XSS_dataset.csv" / "XSS_dataset.csv", encoding="utf-8",
                     low_memory=False)

df_xss["payload_dec"] = df_xss["Sentence"].apply(lambda x: unquote(str(x)))

LABEL_MAP_XSS = {0: LABEL["legitimate"], 1: LABEL["xss"]}

records = [
    make_canonical(sample_id=f"xss_dataset_{i}", label=LABEL_MAP_XSS[lbl], body=pl)
    for i, (lbl, pl) in enumerate(zip(df_xss["Label"], df_xss["payload_dec"]))
]

jsonl_path = CANONICAL_DIR / "xss_dataset.jsonl"
csv_path   = PROCESSED_DIR / "xss_dataset_features.csv"
write_jsonl(records, jsonl_path)
run_extractor(jsonl_path, csv_path)
df_out = load_features(csv_path)

print(f"XSS_dataset.csv | filas={len(df_out):,}")
print(df_out[["sample_id","payload_length","xss_marker_count","html_tag_count","label"]].head(3).to_string())
save_parquet(df_out, "xss_dataset")

  JSONL escrito: xss_dataset.jsonl (13,686 registros)


  OK: 13686 filas escritas en ..\data\processed\xss_dataset_features.csv (0 omitidas)
XSS_dataset.csv | filas=13,686
       sample_id  payload_length  xss_marker_count  html_tag_count  label
0  xss_dataset_0             557                 1               8      0
1  xss_dataset_1              36                 2               2      2
2  xss_dataset_2             233                 0               4      0
  Guardado: xss_dataset.parquet  (13,686 filas, 595 KB)
    label=0 (legitimate          ) :    6,313
    label=2 (xss                 ) :    7,373


WindowsPath('../data/processed/xss_dataset.parquet')

# Seccion 7: data_capec_multilabel.csv

Dataset con schema HTTP completo (38 columnas) y etiquetado multi-label CAPEC.

**Bug corregido (ver `.claude/CANONICAL_REQUEST_NOTES.md` secciones 1-2-4):**
`request_http_request` es SOLO `path?query` (ej. `/blog/xmlrpc.php?rsd`), sin metodo
ni version HTTP. El parser anterior asumia el formato completo `GET /path?qs HTTP/1.1`
y nunca matcheaba -> las 905,069 filas de este dataset (78% del total) tenian las
72 features en cero. El metodo viene de la columna separada `request_http_method`;
`path`/`query` se obtienen partiendo `request_http_request` en el primer `?`.

CanonicalRequest tambien incorpora `userAgent`, `contentType`, `referer`, `cookie`,
`statusCode` (`response_http_status_code`) y el resto de cabeceras disponibles via
`extraHeaders` (Accept, Accept-Language, Accept-Encoding, DNT, Connection, Host,
Origin) -- antes el Grupo 8 (HTTP request) tambien era cero para este dataset porque
`extract_features()` nunca recibia esos campos.

Para filas con multiples labels in-scope, se asigna prioridad: sqli > cmdi > path_traversal > xss.
Filas sin labels de ataque in-scope se clasifican como legitimate. Timestamp: no
disponible, se usa TODAY.


In [8]:
print("Cargando data_capec_multilabel.csv (416 MB)...")
df_cap = pd.read_csv(BASE_DIR / "data_capec_multilabel.csv", low_memory=False)
df_cap = df_cap.drop_duplicates().reset_index(drop=True)
print(f"  Cargado: {len(df_cap):,} filas")

# request_http_request = "path?query" (SIN metodo ni protocolo). El metodo viene
# de request_http_method (columna separada).
RL = df_cap["request_http_request"].fillna("")


def _split_path_qs(s):
    if "?" in s:
        path, qs = s.split("?", 1)
    else:
        path, qs = s, ""
    return path, qs


_parsed = RL.apply(_split_path_qs)
df_cap["_path"]   = _parsed.apply(lambda x: unquote(x[0]))
df_cap["_qs"]     = _parsed.apply(lambda x: unquote(x[1]))
df_cap["_method"] = df_cap["request_http_method"].fillna("GET")

# Label: prioridad sqli > cmdi > path_traversal > xss > legitimate
IN_SCOPE = {
    "66 - SQL Injection":       LABEL["sqli"],
    "88 - OS Command Injection": LABEL["command_injection"],
    "126 - Path Traversal":     LABEL["path_traversal"],
    "242 - Code Injection":     LABEL["xss"],
}
PRIORITY = ["66 - SQL Injection", "88 - OS Command Injection",
            "126 - Path Traversal", "242 - Code Injection"]


def _assign_label(row):
    for col in PRIORITY:
        if col in row and row[col] == 1:
            return IN_SCOPE[col]
    return LABEL["legitimate"]


df_cap["_label"] = df_cap[PRIORITY].apply(_assign_label, axis=1).astype("int8")
print(f"  Distribucion labels:\n{df_cap['_label'].value_counts().sort_index().to_string()}")
n_nonempty = ((df_cap["_path"].str.len() > 0) | (df_cap["_qs"].str.len() > 0)).sum()
print(f"  Filas con _path o _qs no vacios: {n_nonempty:,} / {len(df_cap):,}")

Cargando data_capec_multilabel.csv (416 MB)...


  Cargado: 905,069 filas


  Distribucion labels:
_label
0    615906
1    250230
2     13838
3     18005
4      7090


  Filas con _path o _qs no vacios: 905,068 / 905,069


In [9]:
print("Construyendo registros CanonicalRequest (data_capec)...")

# (atributo df_in, header en extraHeaders, columna origen en df_cap)
HEADER_COLS = [
    ("accept",          "accept",          "request_accept"),
    ("accept_language", "accept-language", "request_accept_language"),
    ("accept_encoding", "accept-encoding", "request_accept_encoding"),
    ("dnt",             "dnt",             "request_do_not_track"),
    ("connection",      "connection",      "request_connection"),
    ("host",            "host",            "request_host"),
    ("origin",          "origin",          "request_origin"),
]

df_in = pd.DataFrame({
    "method":       df_cap["_method"],
    "path":         df_cap["_path"],
    "query":        df_cap["_qs"],
    "body":         df_cap["request_body"].fillna(""),
    "user_agent":   df_cap["request_user_agent"].fillna(""),
    "content_type": df_cap["request_content_type"].fillna(""),
    "referer":      df_cap["request_referer"].fillna(""),
    "cookie":       df_cap["request_cookie"].fillna(""),
    "status":       df_cap["response_http_status_code"].fillna(0).astype(int),
    "label":        df_cap["_label"],
})
for attr, _, col in HEADER_COLS:
    df_in[attr] = df_cap[col].fillna("")

records = []
for i, row in enumerate(df_in.itertuples(index=False)):
    extra = {}
    for attr, hname, _ in HEADER_COLS:
        v = getattr(row, attr)
        if v != "":
            extra[hname] = v
    records.append(make_canonical(
        sample_id=f"capec_{i}", label=row.label,
        method=row.method, path=row.path, query=row.query, body=row.body,
        user_agent=row.user_agent, content_type=row.content_type,
        referer=row.referer, cookie=row.cookie,
        extra_headers=extra, status_code=row.status,
    ))

jsonl_path = CANONICAL_DIR / "data_capec.jsonl"
csv_path   = PROCESSED_DIR / "data_capec_features.csv"
write_jsonl(records, jsonl_path)

print("Extrayendo features data_capec via CLI TS (puede tardar varios minutos para 905K filas)...")
run_extractor(jsonl_path, csv_path)
df_out = load_features(csv_path)

n_nonzero = (df_out["payload_length"] > 0).sum()
print(f"data_capec | filas={len(df_out):,} | payload_length>0: {n_nonzero:,} ({n_nonzero/len(df_out):.1%})")
print(df_out[["sample_id","payload_length","sqli_keyword_count","xss_marker_count",
              "command_separator_count","traversal_sequence_count","label"]].head(5).to_string())
save_parquet(df_out, "data_capec")

del df_cap, df_in, records  # liberar memoria (416 MB)

Construyendo registros CanonicalRequest (data_capec)...


  JSONL escrito: data_capec.jsonl (905,069 registros)
Extrayendo features data_capec via CLI TS (puede tardar varios minutos para 905K filas)...


  OK: 905069 filas escritas en ..\data\processed\data_capec_features.csv (0 omitidas)


data_capec | filas=905,069 | payload_length>0: 905,068 (100.0%)
  sample_id  payload_length  sqli_keyword_count  xss_marker_count  command_separator_count  traversal_sequence_count  label
0   capec_0               1                   0                 0                        0                         0      0
1   capec_1              77                   0                 0                        0                         0      0
2   capec_2               3                   0                 0                        0                         0      0
3   capec_3               1                   0                 0                        0                         0      0
4   capec_4              58                   0                 0                        0                         0      0


  Guardado: data_capec.parquet  (905,069 filas, 13653 KB)
    label=0 (legitimate          ) :  615,906
    label=1 (sqli                ) :  250,230
    label=2 (xss                 ) :   13,838
    label=3 (path_traversal      ) :   18,005
    label=4 (command_injection   ) :    7,090


# Seccion 8: modsec-learn JSON

Dos listas planas de strings:
- `legitimate_dataset.json`: 69,101 unicos (86.4% dupes en crudo)
- `malicious_dataset.json`: 30,544 SQLi, formato `p=<payload>`

CanonicalRequest: el string (URL-decodificado, sin el prefijo `p=`) se asigna a
`query` (`rawPayload = query`; `query_string_length`/`query_param_count` son
significativos porque el registro en si es un query string). Timestamp: no
disponible, se usa TODAY.


In [10]:
with open(BASE_DIR / "modsec-learn" / "legitimate_dataset.json", encoding="utf-8") as f:
    legit_raw = json.load(f)
with open(BASE_DIR / "modsec-learn" / "malicious_dataset.json", encoding="utf-8") as f:
    sqli_raw  = json.load(f)

legit_unique = list(dict.fromkeys(legit_raw))
sqli_unique  = list(dict.fromkeys(sqli_raw))

print(f"legitimate unicos: {len(legit_unique):,}")
print(f"sqli unicos      : {len(sqli_unique):,}")


def _decode_modsec(s):
    s = unquote(str(s))
    if s.startswith("p="):
        s = s[2:]
    return s


records = [
    make_canonical(sample_id=f"modsec_{i}", label=LABEL["legitimate"], query=_decode_modsec(raw))
    for i, raw in enumerate(legit_unique)
]
offset = len(legit_unique)
records += [
    make_canonical(sample_id=f"modsec_{offset + j}", label=LABEL["sqli"], query=_decode_modsec(raw))
    for j, raw in enumerate(sqli_unique)
]

jsonl_path = CANONICAL_DIR / "modsec_learn.jsonl"
csv_path   = PROCESSED_DIR / "modsec_learn_features.csv"
write_jsonl(records, jsonl_path)
run_extractor(jsonl_path, csv_path)
df_out = load_features(csv_path)

print(f"modsec-learn | filas={len(df_out):,}")
print(df_out[["sample_id","payload_length","sqli_keyword_count","union_present","label"]].head(3).to_string())
save_parquet(df_out, "modsec_learn")

legitimate unicos: 69,101
sqli unicos      : 30,544


  JSONL escrito: modsec_learn.jsonl (99,645 registros)


  OK: 99645 filas escritas en ..\data\processed\modsec_learn_features.csv (0 omitidas)


modsec-learn | filas=99,645
  sample_id  payload_length  sqli_keyword_count  union_present  label
0  modsec_0             185                   0              0      0
1  modsec_1              12                   0              0      0
2  modsec_2             382                   0              0      0
  Guardado: modsec_learn.parquet  (99,645 filas, 3901 KB)
    label=0 (legitimate          ) :   69,101
    label=1 (sqli                ) :   30,544


WindowsPath('../data/processed/modsec_learn.parquet')

# Seccion 9: PT Wordlists (Dp.txt)

Solo se procesa `Dp.txt` (1,166 payloads de Path Traversal reales para Windows).
`Deep-Travelsal.txt` contiene templates con placeholder {FILE}: se omite en este paso
por requerir expansion de archivo objetivo antes de ser valido como muestra de ataque.
CanonicalRequest: cada linea (URL-decodificada) se asigna a `path`
(`rawPayload = path`; `path_depth`/`path_separator_count`/`absolute_path_indicator`
son significativos). Todos los samples son label=3 (path_traversal). Timestamp: TODAY.


In [11]:
PT_DIR = BASE_DIR / "omurugur Path_Travelsal_Payload_List master Payload"
dp_lines = (PT_DIR / "Dp.txt").read_text(encoding="utf-8", errors="replace").splitlines()
dp_lines = [l.strip() for l in dp_lines if l.strip()]

records = [
    make_canonical(sample_id=f"pt_wordlist_{i}", label=LABEL["path_traversal"], path=unquote(line))
    for i, line in enumerate(dp_lines)
]

jsonl_path = CANONICAL_DIR / "pt_wordlists.jsonl"
csv_path   = PROCESSED_DIR / "pt_wordlists_features.csv"
write_jsonl(records, jsonl_path)
run_extractor(jsonl_path, csv_path)
df_out = load_features(csv_path)

print(f"PT wordlists | filas={len(df_out):,}")
print(df_out[["sample_id","traversal_sequence_count","path_separator_count","sensitive_file_target","label"]].head(4).to_string())
save_parquet(df_out, "pt_wordlists")

  JSONL escrito: pt_wordlists.jsonl (1,166 registros)


  OK: 1166 filas escritas en ..\data\processed\pt_wordlists_features.csv (0 omitidas)
PT wordlists | filas=1,166
       sample_id  traversal_sequence_count  path_separator_count  sensitive_file_target  label
0  pt_wordlist_0                         1                     3                      1      3
1  pt_wordlist_1                         2                     4                      1      3
2  pt_wordlist_2                         3                     5                      1      3
3  pt_wordlist_3                         4                     6                      1      3
  Guardado: pt_wordlists.parquet  (1,166 filas, 68 KB)
    label=3 (path_traversal      ) :    1,166


WindowsPath('../data/processed/pt_wordlists.parquet')

# Seccion 10: OWASP ModSecurity Logs

30 dias de logs (379.5 MB, 142,705 transacciones). Se parsean los bloques de transaccion
para extraer method, path, query string, body (seccion C, primeros 2000 caracteres),
cabeceras (seccion B), status (seccion F), IP y timestamp real (seccion A).
Solo se incluyen las 56,504 transacciones in-scope (sqli, xss, path_traversal, cmdi).

CanonicalRequest: `path`/`query` decodificados; `body` se deja sin decodificar (igual
que `request_body` en data_capec) -- si hay body, `rawPayload = body` (mayor prioridad
en `deriveRawPayload`), lo cual es razonable porque ModSecurity solo registra la
seccion C cuando la peticion tuvo cuerpo (tipicamente el vector real en ataques POST).
Todas las cabeceras no-request-line de la seccion B se vuelcan a `extraHeaders`
(claves en minusculas), de donde se derivan `userAgent`, `contentType`, `referer`,
`cookie`, `authorization_length` y `unusual_headers_count` (mismo set de 12 headers
estandar que el parser anterior).

Las features temporales (Grupo 9) se calculan por separado con `add_temporal_features()`
usando IP+timestamp+path+status, ya que el extractor TS es una funcion pura por-request
y no tiene estado de sesion.


In [12]:
OWASP_DIR  = BASE_DIR / "owasp"
log_files  = sorted(OWASP_DIR.rglob("*.anon.log"))

_TS_PAT  = re.compile(r'\[(\d{2}/\w+/\d{4}:\d{2}:\d{2}:\d{2} [+\-]\d{4})\]')
_REQ_PAT = re.compile(r'^(GET|POST|PUT|DELETE|HEAD|OPTIONS|PATCH|CONNECT|TRACE)\s+(\S+)\s+HTTP', re.M)
_ST_PAT  = re.compile(r'HTTP/\S+\s+(\d{3})', re.M)
_IP_PAT  = re.compile(r'^\[\S+\s+\S+\]\s+\S+\s+(\S+)', re.M)


def classify_owasp(tags, rule_ids):
    if "attack-sqli" in tags:                                                return "sqli"
    if "attack-xss"  in tags:                                                return "xss"
    if any(r in rule_ids for r in ["930110","930120","930130"]) \
       or "attack-lfi" in tags or "attack-rfi" in tags:                      return "path_traversal"
    if any(r in rule_ids for r in ["932150","932160"]) \
       or "attack-rce" in tags:                                              return "command_injection"
    return None


records_meta = []  # ip/timestamp/path/status para add_temporal_features
records_can  = []  # CanonicalRequest

for lf in log_files:
    # Extrae fecha del directorio para fallback de timestamp
    dir_date = lf.parent.name  # e.g. '15-Aug-2025'
    try:
        fallback_ts = pd.Timestamp(dir_date)
    except Exception:
        fallback_ts = TODAY

    with open(lf, "r", encoding="utf-8", errors="replace") as fh:
        content = fh.read()

    for tx in re.split(r"(?=--[a-f0-9]+-A--)", content):
        if not tx.strip(): continue

        sec_h = re.search(r"-H--\n(.*?)(?=--[a-f0-9]+-Z--|$)", tx, re.DOTALL)
        if not sec_h: continue
        h_text = sec_h.group(1)

        tags     = set(re.findall(r'\[tag "([^"]+)"\]', h_text))
        rule_ids = set(re.findall(r'\[id "(\d+)"\]', h_text))
        label_str = classify_owasp(tags, rule_ids)
        if label_str is None: continue

        # Timestamp desde seccion A
        sec_a = re.search(r"-A--\n(.+)", tx)
        ts = fallback_ts
        if sec_a:
            tm = _TS_PAT.search(sec_a.group(1))
            if tm:
                try:
                    ts = pd.to_datetime(tm.group(1), format="%d/%b/%Y:%H:%M:%S %z").tz_convert("UTC").tz_localize(None)
                except Exception:
                    pass

        # Request line + cabeceras desde seccion B
        sec_b = re.search(r"-B--\n(.*?)(?=--[a-f0-9]+-[A-Z]--|$)", tx, re.DOTALL)
        method, path, qs_str = "", "", ""
        headers = {}
        if sec_b:
            b_text = sec_b.group(1)
            rm = _REQ_PAT.search(b_text)
            if rm:
                method = rm.group(1)
                full   = rm.group(2)
                path, qs_str = (full.split("?",1) if "?" in full else (full, ""))
            for line in b_text.splitlines()[1:]:
                if ":" in line:
                    hname, _, hval = line.partition(":")
                    hname = hname.strip().lower()
                    if hname:
                        headers[hname] = hval.strip()

        # Body desde seccion C
        sec_c = re.search(r"-C--\n(.*?)(?=--[a-f0-9]+-[A-Z]--|$)", tx, re.DOTALL)
        body_str = sec_c.group(1)[:2000] if sec_c else ""

        # Status desde seccion F
        sec_f = re.search(r"-F--\n(.*?)(?=--[a-f0-9]+-[A-Z]--|$)", tx, re.DOTALL)
        status_val = 0
        if sec_f:
            sm = _ST_PAT.search(sec_f.group(1))
            if sm: status_val = int(sm.group(1))

        # IP desde seccion A
        ip_str = "0.0.0.0"
        if sec_a:
            ipm = _IP_PAT.search(sec_a.group(1))
            if ipm: ip_str = ipm.group(1)

        path_dec = unquote(path)
        qs_dec   = unquote(qs_str)

        records_meta.append({
            "label_str": label_str, "timestamp": ts,
            "path": path_dec, "status": status_val, "ip": ip_str,
        })
        records_can.append(make_canonical(
            sample_id=f"owasp_{len(records_can)}",
            label=LABEL[label_str], timestamp=ts,
            method=method, path=path_dec, query=qs_dec, body=body_str,
            user_agent=headers.get("user-agent",""),
            content_type=headers.get("content-type",""),
            referer=headers.get("referer",""),
            cookie=headers.get("cookie",""),
            extra_headers=headers,
            status_code=status_val,
        ))

df_ow = pd.DataFrame(records_meta)
print(f"OWASP in-scope transacciones: {len(df_ow):,}")
print(df_ow["label_str"].value_counts().to_string())

OWASP in-scope transacciones: 56,504
label_str
path_traversal       49849
sqli                  4064
command_injection     2331
xss                    260


In [13]:
jsonl_path = CANONICAL_DIR / "owasp_logs.jsonl"
csv_path   = PROCESSED_DIR / "owasp_logs_features.csv"
write_jsonl(records_can, jsonl_path)
run_extractor(jsonl_path, csv_path)
df_out = load_features(csv_path)

# Features temporales por IP (ventanas deslizantes) -- ver Seccion 2
print("Calculando features temporales OWASP...")
df_out = add_temporal_features(
    df_out, df_ow,
    ts_col="timestamp", ip_col="ip",
    endpoint_col="path", status_col="status",
)

print(f"OWASP | filas={len(df_out):,}")
print(df_out[["sample_id","traversal_sequence_count","sqli_keyword_count",
              "req_count_60s","label"]].head(3).to_string())
save_parquet(df_out, "owasp_logs")

del df_ow, records_can, records_meta

  JSONL escrito: owasp_logs.jsonl (56,504 registros)


  OK: 56504 filas escritas en ..\data\processed\owasp_logs_features.csv (0 omitidas)


Calculando features temporales OWASP...


OWASP | filas=56,504
  sample_id  traversal_sequence_count  sqli_keyword_count  req_count_60s  label
0   owasp_0                         0                   0            0.0      3
1   owasp_1                         0                   0            0.0      3
2   owasp_2                         0                   0            0.0      3
  Guardado: owasp_logs.parquet  (56,504 filas, 1313 KB)
    label=1 (sqli                ) :    4,064
    label=2 (xss                 ) :      260
    label=3 (path_traversal      ) :   49,849
    label=4 (command_injection   ) :    2,331


# Seccion 11: RussellMitchell (legitimate only)

Access logs Apache CLF del escenario WordPress. Se usa solo como fuente de trafico legitimo:
- `.log.1`, `.log.3`, `.log.4`: trafico sin etiquetar (todos asumidos legitimate)
- `.log.2`: solo las lineas sin label (legitimate confirmado)

CanonicalRequest: `path`/`query` decodificados desde la request line CLF, `userAgent`
y `statusCode` desde las columnas correspondientes. Timestamp: extraido del formato
CLF `[23/Jan/2022:10:15:30 +0000]`. Las features temporales (Grupo 9) se calculan
igual que para OWASP.


In [14]:
RM_LOG_DIR    = BASE_DIR / "russellmitchell" / "gather" / "intranet_server" / "logs" / "apache2"
RM_LABELS_DIR = BASE_DIR / "russellmitchell" / "labels" / "intranet_server" / "logs" / "apache2"

_CLF_PAT  = re.compile(
    r'(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+) \S+" (\d+) \S+'
    r'(?: "[^"]*" "([^"]*)")?'  # referer + UA opcionalmente
)
_MONTH = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
          "Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}


def _parse_clf_ts(s):
    # '23/Jan/2022:10:15:30 +0000'
    try:
        d, rest = s.split(":", 1)
        day, mon, yr = d.split("/")
        h, mi, sec_tz = rest.split(":", 2)
        sec = sec_tz.split(" ")[0]
        return pd.Timestamp(int(yr), _MONTH[mon], int(day), int(h), int(mi), int(sec))
    except Exception:
        return TODAY


# Cargar lineas etiquetadas de log.2 para filtrar las labeled (recon/attack)
label_file  = RM_LABELS_DIR / "intranet.smith.russellmitchell.com-access.log.2"
labeled_lines = set()
with open(label_file) as lf:
    for line in lf:
        obj = json.loads(line)
        labeled_lines.add(obj["line"])

records_meta = []
records_can  = []

for fname in [
    "intranet.smith.russellmitchell.com-access.log.2",
    "intranet.smith.russellmitchell.com-access.log.1",
    "intranet.smith.russellmitchell.com-access.log.3",
    "intranet.smith.russellmitchell.com-access.log.4",
]:
    fpath = RM_LOG_DIR / fname
    if not fpath.exists(): continue
    with open(fpath, encoding="utf-8", errors="replace") as fh:
        for lineno, line in enumerate(fh, start=1):
            # Para log.2: saltar lineas con labels de ataque
            if fname.endswith(".log.2") and lineno in labeled_lines:
                continue
            m = _CLF_PAT.match(line.strip())
            if not m: continue
            ip, ts_str, method, full_path, status = (
                m.group(1), m.group(2), m.group(3), m.group(4), int(m.group(5))
            )
            ua_str = m.group(6) or ""
            path, qs_str = (full_path.split("?",1) if "?" in full_path else (full_path, ""))
            ts = _parse_clf_ts(ts_str)
            path_dec = unquote(path)
            qs_dec   = unquote(qs_str)

            records_meta.append({"timestamp": ts, "path": path_dec, "status": status, "ip": ip})
            records_can.append(make_canonical(
                sample_id=f"russellmitchell_{len(records_can)}",
                label=LABEL["legitimate"], timestamp=ts,
                method=method, path=path_dec, query=qs_dec,
                user_agent=ua_str, status_code=status,
            ))

df_rm = pd.DataFrame(records_meta)
print(f"RussellMitchell legitimate | filas={len(df_rm):,}")

jsonl_path = CANONICAL_DIR / "russellmitchell.jsonl"
csv_path   = PROCESSED_DIR / "russellmitchell_features.csv"
write_jsonl(records_can, jsonl_path)
run_extractor(jsonl_path, csv_path)
df_out = load_features(csv_path)

df_out = add_temporal_features(
    df_out, df_rm,
    ts_col="timestamp", ip_col="ip",
    endpoint_col="path", status_col="status",
)

print(f"RussellMitchell | filas={len(df_out):,}")
save_parquet(df_out, "russellmitchell")

del df_rm, records_can, records_meta

RussellMitchell legitimate | filas=3,435
  JSONL escrito: russellmitchell.jsonl (3,435 registros)
  OK: 3435 filas escritas en ..\data\processed\russellmitchell_features.csv (0 omitidas)


RussellMitchell | filas=3,435
  Guardado: russellmitchell.parquet  (3,435 filas, 102 KB)
    label=0 (legitimate          ) :    3,435


# Seccion 12: Resumen de parquets generados

Verificacion de que todos los archivos existen, tienen el schema correcto y
no hay labels inesperados ni columnas faltantes. Incluye validacion especifica
de que `data_capec` ya no tiene las 72 features en cero (bug corregido en Seccion 7).


In [15]:
import pyarrow.parquet as pq

expected_files = [
    "payloads_csv", "payload_full", "command_injection",
    "xss_dataset", "data_capec", "modsec_learn",
    "pt_wordlists", "owasp_logs", "russellmitchell",
]

inv_label = {v: k for k, v in LABEL.items()}
total_rows = 0

print(f"{'Archivo':<25} {'Filas':>8}  {'MB':>6}  {'Cols':>5}  Label distribution")
print("-" * 100)

for name in expected_files:
    fpath = PROCESSED_DIR / f"{name}.parquet"
    if not fpath.exists():
        print(f"  FALTA: {name}.parquet")
        continue
    df_tmp = pd.read_parquet(fpath)
    size_mb = fpath.stat().st_size / (1024**2)
    n_rows  = len(df_tmp)
    n_cols  = len(df_tmp.columns)
    total_rows += n_rows
    missing_feats = [c for c in FEATURE_COLS if c not in df_tmp.columns]
    dist = df_tmp["label"].value_counts().sort_index()
    dist_str = "  ".join(f"{inv_label.get(int(k),k)}:{v:,}" for k, v in dist.items())
    warn = f" [FALTAN COLS: {missing_feats}]" if missing_feats else ""
    print(f"  {name:<23} {n_rows:>8,}  {size_mb:>6.1f}  {n_cols:>5}  {dist_str}{warn}")

print()
print(f"Total filas en processed/: {total_rows:,}")
print(f"Schema esperado: sample_id + {len(FEATURE_COLS)} features + label + timestamp = {3 + len(FEATURE_COLS)} columnas")

# Validacion especifica: data_capec ya no debe tener las 72 features en cero (Seccion 7)
cap_path = PROCESSED_DIR / "data_capec.parquet"
if cap_path.exists():
    df_cap_check = pd.read_parquet(cap_path, columns=["payload_length", "sqli_keyword_count", "label"])
    zero_rows = (df_cap_check["payload_length"] == 0).sum()
    print(f"\ndata_capec: filas con payload_length==0: {zero_rows:,} / {len(df_cap_check):,} "
          f"({zero_rows/len(df_cap_check):.1%})")

Archivo                      Filas      MB   Cols  Label distribution
----------------------------------------------------------------------------------------------------
  payloads_csv              42,671     1.5     75  legitimate:28,068  xss:14,603


  payload_full              31,067     0.8     75  legitimate:19,304  sqli:10,852  xss:532  path_traversal:290  command_injection:89
  command_injection          2,059     0.1     75  legitimate:1,581  command_injection:478
  xss_dataset               13,686     0.6     75  legitimate:6,313  xss:7,373


  data_capec               905,069    13.3     75  legitimate:615,906  sqli:250,230  xss:13,838  path_traversal:18,005  command_injection:7,090
  modsec_learn              99,645     3.8     75  legitimate:69,101  sqli:30,544
  pt_wordlists               1,166     0.1     75  path_traversal:1,166
  owasp_logs                56,504     1.3     75  sqli:4,064  xss:260  path_traversal:49,849  command_injection:2,331
  russellmitchell            3,435     0.1     75  legitimate:3,435

Total filas en processed/: 1,155,302
Schema esperado: sample_id + 72 features + label + timestamp = 75 columnas

data_capec: filas con payload_length==0: 1 / 905,069 (0.0%)
